# **Import**

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor

# **Data Load**

In [ ]:
cd /content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data

/content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data


In [ ]:
train_df = pd.read_csv('./train.csv')
pair_df = pd.read_csv('./pairs.csv')

# **Build Model per Best lag**

In [ ]:
monthly = (
    train_df
    .groupby(['item_id', 'year', 'month'], as_index=False)['value']
    .sum()
)
monthly['ym'] = pd.to_datetime(monthly['year'].astype(str) + '-' + monthly['month'].astype(str))
pivot = monthly.pivot(index='item_id', columns='ym', values='value')
pivot = pivot.fillna(0)
pivot

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YSYHGLQK,0.0,543.0,766.0,1108.0,859.0,1426.0,2413.0,638.0,0.0,1199.0,...,188.0,541.0,696.0,8710.0,3175.0,2624.0,0.0,182.0,2128.0,10651.0
ZCELVYQU,373859.0,59900.0,31158.0,594407.0,648232.0,496737.0,210179.0,0.0,70748.0,15512.0,...,0.0,609803.0,23712.0,654630.0,4496.0,1177300.0,1187539.0,26434.0,115631.0,270262.0
ZGJXVMNI,1154724.0,1337622.0,1662893.0,1561647.0,1603223.0,1641942.0,1815161.0,1546959.0,1536799.0,1496906.0,...,3168505.0,3059865.0,1579976.0,1413293.0,3038078.0,2915914.0,3565526.0,3020051.0,2412781.0,2458481.0


In [ ]:
max_self_lag = 3
ma_windows = [3, 6]
std_windows = [3, 6]

def make_features_by_best_lag(filtered_pair_df):
    all_rows = []  # 함수 안에서 초기화
    for _, row in tqdm(filtered_pair_df.iterrows(), total=len(filtered_pair_df)):
        leader = row['leading_item_id']
        follower = row['following_item_id']
        best_lag = int(row['best_lag'])

        s_lead = pivot.loc[leader]
        s_follow = pivot.loc[follower]

        for ym in pivot.columns:
            rec = {'month': ym, 'leader_item': leader, 'follower_item': follower}

            # Leader lag 주변 추가
            for lag_shift in [-1, 0, 1]:
                lag = best_lag + lag_shift
                if lag <= 0:
                    continue
                rec[f'leader_lag{lag}'] = s_lead.get(ym - pd.DateOffset(months=lag), np.nan)

            # Follower lag
            for l in range(1, max_self_lag + 1):
                rec[f'follower_lag{l}'] = s_follow.get(ym - pd.DateOffset(months=l), np.nan)

            # 이동평균
            for w in ma_windows:
                rec[f'leader_ma{w}'] = s_lead[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                    if len(s_lead[:ym - pd.DateOffset(months=1)]) >= w else np.nan
                rec[f'follower_ma{w}'] = s_follow[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                    if len(s_follow[:ym - pd.DateOffset(months=1)]) >= w else np.nan

            # STD Feature
            for w in std_windows:
                rec[f'leader_std{w}'] = s_lead[:ym - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                    if len(s_lead[:ym - pd.DateOffset(months=1)]) >= w else np.nan
                rec[f'follower_std{w}'] = s_follow[:ym - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                    if len(s_follow[:ym - pd.DateOffset(months=1)]) >= w else np.nan

            # DIFF Feature
            for l in range(1, max_self_lag):
                # Leader diff
                if not pd.isna(rec.get(f'leader_lag{l}')) and not pd.isna(rec.get(f'leader_lag{l+1}')):
                    rec[f'leader_diff{l}'] = rec[f'leader_lag{l}'] - rec[f'leader_lag{l+1}']
                # Follower diff
                if not pd.isna(rec.get(f'follower_lag{l}')) and not pd.isna(rec.get(f'follower_lag{l+1}')):
                    rec[f'follower_diff{l}'] = rec[f'follower_lag{l}'] - rec[f'follower_lag{l+1}']

            # Month number
            rec['month_num'] = ym.month

            # Target
            rec['target'] = s_follow.get(ym, np.nan)

            all_rows.append(rec)

    feature_df = pd.DataFrame(all_rows)
    feature_df = feature_df.sort_values(['leader_item', 'follower_item', 'month']).reset_index(drop=True)
    return feature_df


In [ ]:
feature_df_1 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 1])
feature_df_2 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 2])
feature_df_3 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 3])
feature_df_4 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 4])
feature_df_5 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 5])
feature_df_6 = make_features_by_best_lag(pair_df[pair_df['best_lag'] == 6])

100%|██████████| 272/272 [00:50<00:00,  5.36it/s]


In [ ]:
feature_df_list = [feature_df_1, feature_df_2, feature_df_3,
                   feature_df_4, feature_df_5, feature_df_6]

In [ ]:
results_each_lag = []

i = 0
for feature_df in feature_df_list:
    i += 1

    train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
    val_month = pd.Timestamp('2025-07-01')

    train_df_features = feature_df[feature_df['month'] <= train_cutoff].copy()
    val_df_features = feature_df[feature_df['month'] == val_month].copy()

    x_cols = [c for c in train_df_features.columns if c not in ['month','leader_item','follower_item','target']]

    train_x = train_df_features[x_cols]
    train_y = train_df_features['target']
    val_x = val_df_features[x_cols]
    val_y = val_df_features['target']

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

    pred_y = model.predict(val_x)

    results_each_lag.append({
        'best_lag': i,
        'MAE': mean_absolute_error(val_y, pred_y),
        'RMSE': mean_squared_error(val_y, pred_y)**0.5
    })

In [ ]:
result_df = pd.DataFrame(results_each_lag)
print(result_df['MAE'].mean(), result_df['RMSE'].mean())
result_df

1665604.3437010068 4431104.201071757


,best_lag,MAE,RMSE
0,1,2.190621e+06,5.432287e+06
1,2,1.973854e+06,5.780460e+06
2,3,1.938206e+06,4.476823e+06
3,4,1.351984e+06,3.558683e+06
4,5,1.196565e+06,3.103433e+06
5,6,1.342397e+06,4.234939e+06


# **Prediction**

In [ ]:
def make_pred_features_by_best_lag(filtered_pair_df):
    pred_month = pd.Timestamp('2025-08-01')
    all_rows = []

    for _, row in filtered_pair_df.iterrows():
        leader = row['leading_item_id']
        follower = row['following_item_id']
        best_lag = int(row['best_lag'])

        s_lead = pivot.loc[leader]
        s_follow = pivot.loc[follower]

        rec = {'month': pred_month, 'leader_item': leader, 'follower_item': follower}

        # Leader lag 주변
        for lag_shift in [-1, 0, 1]:
            lag = best_lag + lag_shift
            if lag <= 0:
                continue
            rec[f'leader_lag{lag}'] = s_lead.get(pred_month - pd.DateOffset(months=lag), np.nan)

        # Follower lag
        for l in range(1, max_self_lag + 1):
            rec[f'follower_lag{l}'] = s_follow.get(pred_month - pd.DateOffset(months=l), np.nan)

        # 이동평균
        for w in ma_windows:
            rec[f'leader_ma{w}'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                if len(s_lead[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan
            rec[f'follower_ma{w}'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                if len(s_follow[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan

        # STD Feature
        for w in std_windows:
            rec[f'leader_std{w}'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                if len(s_lead[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan
            rec[f'follower_std{w}'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                if len(s_follow[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan

        # DIFF Feature
        for l in range(1, max_self_lag):
            if not pd.isna(rec.get(f'leader_lag{l}')) and not pd.isna(rec.get(f'leader_lag{l+1}')):
                rec[f'leader_diff{l}'] = rec[f'leader_lag{l}'] - rec[f'leader_lag{l+1}']
            if not pd.isna(rec.get(f'follower_lag{l}')) and not pd.isna(rec.get(f'follower_lag{l+1}')):
                rec[f'follower_diff{l}'] = rec[f'follower_lag{l}'] - rec[f'follower_lag{l+1}']

        rec['month_num'] = pred_month.month
        rec['target'] = np.nan

        all_rows.append(rec)

    # 반환
    return pd.DataFrame(all_rows)

In [ ]:
all_pred_dfs = []

for curr_lag in range(1, 7):
    feature_df = make_features_by_best_lag(pair_df[pair_df['best_lag'] == curr_lag])

    train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
    val_month = pd.Timestamp('2025-07-01')

    train_df_features = feature_df[feature_df['month'] <= train_cutoff].copy()
    val_df_features = feature_df[feature_df['month'] == val_month].copy()

    x_cols = [c for c in train_df_features.columns if c not in ['month','leader_item','follower_item','target']]

    train_x = train_df_features[x_cols]
    train_y = train_df_features['target']
    val_x = val_df_features[x_cols]
    val_y = val_df_features['target']

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

    pred_df = make_pred_features_by_best_lag(pair_df[pair_df['best_lag'] == curr_lag])
    pred_df['value'] = model.predict(pred_df[x_cols])

    all_pred_dfs.append(pred_df[['leader_item','follower_item','value']])

In [ ]:
submission = pd.concat(all_pred_dfs, axis=0).sort_values(['leader_item','follower_item']).reset_index(drop=True)
submission.rename(columns={'leader_item': 'leading_item_id', 'follower_item': 'following_item_id'}, inplace=True)
submission.head()

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,1.068985e+05
1,AANGBULD,DEWLVASR,5.540930e+05
2,AANGBULD,DNMPSKTB,4.084345e+06
3,AANGBULD,EVBVXETX,5.693627e+06
4,AANGBULD,FTSVTTSR,2.346554e+05


In [ ]:
submission.to_csv('submission2.csv', index=False)

pd.read_csv('submission2.csv')

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,1.068985e+05
1,AANGBULD,DEWLVASR,5.540930e+05
2,AANGBULD,DNMPSKTB,4.084345e+06
3,AANGBULD,EVBVXETX,5.693627e+06
4,AANGBULD,FTSVTTSR,2.346554e+05
...,...,...,...
1420,ZXERAXWP,DBWLZWNK,2.341221e+05
1421,ZXERAXWP,FITUEHWN,1.118813e+05
1422,ZXERAXWP,MIRCVAMV,5.180373e+04
1423,ZXERAXWP,UIFPPCLR,9.872719e+04
